# Submission 4

In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.6-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatib

In [2]:
%%writefile /kaggle/working/my_agent.py

from __future__ import annotations

import math
import random
import time
import hashlib
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Any, Optional

import numpy as np

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState


INFINITY = np.iinfo(np.int32).max
SIMPLE_ACTION_IDS = {1, 2, 3, 4, 5, 7}
EDGE_DTYPE = np.dtype(
    [
        ("group", "i4"),
        ("result", "i4"),   # 1 success, -1 failed, 0 untested
        ("target", "U64"),  # target node id
        ("distance", "i4"),
        ("errors", "i4"),
    ]
)


def normalize_action_id(action: Any) -> Optional[int]:
    if action is None:
        return None
    try:
        if hasattr(action, "value"):
            value = int(action.value)
            if value > 0:
                return value
    except Exception:
        pass
    try:
        if hasattr(action, "id"):
            value = int(action.id)
            if value > 0:
                return value
    except Exception:
        pass
    try:
        value = int(action)
        if value > 0:
            return value
    except Exception:
        pass
    return None


def extract_latest_grid(frame_data: FrameData) -> tuple[np.ndarray, int]:
    arr = np.asarray(frame_data.frame, dtype=np.uint8)
    if arr.ndim == 2:
        return arr, 1
    if arr.ndim >= 3:
        return np.asarray(arr[-1], dtype=np.uint8), int(arr.shape[0])
    return np.zeros((64, 64), dtype=np.uint8), 0


@dataclass
class NodeInfo:
    name: str
    total_candidates: int
    num_groups: int
    group2remaining_candidate_ids: list[set[int]]
    error_threshold: int = 3
    closed: bool = False
    distance: int = INFINITY
    edge_data: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        self.group2remaining_candidate_ids = [set(s) for s in self.group2remaining_candidate_ids]
        self.edge_data = np.zeros(self.total_candidates, dtype=EDGE_DTYPE)
        for group_id, candidate_ids in enumerate(self.group2remaining_candidate_ids):
            if candidate_ids:
                self.edge_data["group"][list(candidate_ids)] = int(group_id)

    def has_open_group(self, group_id: int) -> bool:
        for gid in range(min(group_id, self.num_groups - 1) + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: int, target_node: Optional[str] = None) -> bool:
        if edge_idx < 0 or edge_idx >= self.total_candidates:
            return False

        current_result = int(self.edge_data["result"][edge_idx])
        if current_result != 0:
            if success == 1 and target_node and str(target_node) != str(self.edge_data["target"][edge_idx]):
                pass
            else:
                return False

        edge_group_id = int(self.edge_data["group"][edge_idx])

        if success == -1:
            self.edge_data["errors"][edge_idx] += 1
            if self.edge_data["errors"][edge_idx] < self.error_threshold:
                return False
            self.edge_data["errors"][edge_idx] = 0
            new_group_id = edge_group_id + 1
            self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
            if new_group_id <= self.num_groups - 1:
                self.edge_data["group"][edge_idx] = new_group_id
                self.group2remaining_candidate_ids[new_group_id].add(edge_idx)
                return False
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
            return True

        self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)

        if success == 1:
            self.edge_data["result"][edge_idx] = 1
            self.edge_data["target"][edge_idx] = str(target_node or "")
            self.edge_data["distance"][edge_idx] = -1
        else:
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
        return True


class GraphExplorer:
    def __init__(self, n_groups: int = 5):
        self._n_groups = max(1, int(n_groups))
        self.reset()

    def reset(self) -> None:
        self._nodes: dict[str, NodeInfo] = {}
        self._G: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._G_rev: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._frontier: set[str] = set()
        self._dist: dict[str, int] = {}
        self._next: dict[str, tuple[int, str]] = {}
        self._active_group = 0
        self._empty = True
        self.suspicious_transitions: dict[tuple[str, int, str], int] = {}
        self.suspicious_transitions_threshold = 3

    @property
    def active_group(self) -> int:
        return self._active_group

    @property
    def frontier(self) -> set[str]:
        return self._frontier

    def distance(self, node: str) -> int:
        return int(self._dist.get(node, INFINITY))

    def has_node(self, node: str) -> bool:
        return str(node) in self._nodes

    def initialize(self, start_node: str, num_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        self.ensure_node(start_node, num_candidates, group2remaining_candidate_ids)
        self._maybe_advance_group(start_node)

    def ensure_node(self, node: str, n_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        if node in self._nodes:
            return
        self._nodes[node] = NodeInfo(
            name=node,
            total_candidates=int(n_candidates),
            num_groups=self._n_groups,
            group2remaining_candidate_ids=group2remaining_candidate_ids,
        )
        self._G[node] = set()
        self._G_rev[node] = set()
        self._empty = False
        if self._nodes[node].has_open_group(self._active_group):
            self._frontier.add(node)
        else:
            self._close_node(node)

    def _remove_edge(self, node: str, edge_idx: int) -> None:
        for existing_idx, target in list(self._G.get(node, set())):
            if int(existing_idx) == int(edge_idx):
                self._G[node].discard((existing_idx, target))
                self._G_rev[target].discard((existing_idx, node))

    def _close_node(self, node: str) -> None:
        info = self._nodes[node]
        if info.closed:
            return
        info.closed = True
        self._frontier.discard(node)
        self._rebuild_distances()

    def _rebuild_distances(self) -> None:
        self._dist.clear()
        self._next.clear()

        dq = deque()
        for node, info in self._nodes.items():
            info.distance = INFINITY
            self._dist[node] = INFINITY

        for node in self._frontier:
            self._dist[node] = 0
            self._nodes[node].distance = 0
            dq.append(node)

        while dq:
            v = dq.popleft()
            v_dist = self._dist.get(v, INFINITY)
            for edge_idx, u in self._G_rev.get(v, ()):
                cand_dist = v_dist + 1
                self._nodes[u].edge_data["distance"][edge_idx] = cand_dist
                if cand_dist < self._dist.get(u, INFINITY):
                    self._dist[u] = cand_dist
                    self._nodes[u].distance = cand_dist
                    self._next[u] = (edge_idx, v)
                    dq.append(u)

    def _maybe_advance_group(self, current_node: str) -> None:
        while self.distance(current_node) >= INFINITY and self._active_group < self._n_groups - 1:
            self._active_group += 1
            self._frontier.clear()
            for node, info in self._nodes.items():
                info.closed = False
                if info.has_open_group(self._active_group):
                    self._frontier.add(node)
            self._rebuild_distances()

    def record_test(
        self,
        node: str,
        edge_idx: int,
        success: int,
        target_node: Optional[str] = None,
        target_num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[list[set[int]]] = None,
        suspicious_transition: bool = False,
    ) -> None:
        if node not in self._nodes:
            return

        if suspicious_transition and success == 1 and target_node is not None:
            key = (str(node), int(edge_idx), str(target_node))
            self.suspicious_transitions[key] = self.suspicious_transitions.get(key, 0) + 1
            if self.suspicious_transitions[key] < self.suspicious_transitions_threshold:
                return

        node_info = self._nodes[node]
        wrote = node_info.record_test(int(edge_idx), int(success), target_node)
        if not wrote:
            return

        if success == 1 and target_node is not None:
            self._remove_edge(node, edge_idx)
            if target_node not in self._nodes:
                if target_num_candidates is None or group2remaining_candidate_ids is None:
                    return
                self.ensure_node(target_node, int(target_num_candidates), group2remaining_candidate_ids)
            self._G[node].add((int(edge_idx), str(target_node)))
            self._G_rev[target_node].add((int(edge_idx), str(node)))

            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)

            if self._nodes[target_node].has_open_group(self._active_group):
                self._rebuild_distances()
            else:
                self._close_node(target_node)
                self._maybe_advance_group(target_node)
        else:
            self._remove_edge(node, edge_idx)
            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)
                self._maybe_advance_group(node)

    def open_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        if not node_info.has_open_group(self._active_group):
            return []
        out: list[int] = []
        for gid in range(self._active_group + 1):
            out.extend(sorted(node_info.group2remaining_candidate_ids[gid]))
        return out

    def successful_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        lowest_dist = self.distance(node)
        if lowest_dist >= INFINITY:
            return []
        return [
            idx
            for idx, row in enumerate(node_info.edge_data)
            if int(row["result"]) == 1
            and int(row["group"]) <= self._active_group
            and int(row["distance"]) <= lowest_dist
        ]

    def edge_distance(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return INFINITY
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return INFINITY
        return int(node_info.edge_data["distance"][edge_idx])


@dataclass
class BetaCounter:
    success: float = 1.0
    failure: float = 1.0

    def mean(self) -> float:
        denom = self.success + self.failure
        return float(self.success / denom) if denom > 0 else 0.5

    @property
    def count(self) -> float:
        return max(0.0, self.success + self.failure - 2.0)

    def update(self, success_inc: float, failure_inc: float) -> None:
        self.success += max(0.0, float(success_inc))
        self.failure += max(0.0, float(failure_inc))


class FeatureMemory:
    def __init__(self) -> None:
        self._stats: dict[tuple[Any, ...], BetaCounter] = {}

    def _get(self, key: tuple[Any, ...]) -> BetaCounter:
        if key not in self._stats:
            self._stats[key] = BetaCounter()
        return self._stats[key]

    def mean(self, key: tuple[Any, ...]) -> float:
        return self._get(key).mean()

    def count(self, key: tuple[Any, ...]) -> float:
        return self._get(key).count

    def update(self, key: tuple[Any, ...], success_inc: float, failure_inc: float) -> None:
        self._get(key).update(success_inc, failure_inc)

    def _candidate_weighted_keys(self, candidate: dict[str, Any]) -> list[tuple[tuple[Any, ...], float]]:
        feats = candidate.get("features") or {}
        kind = str(candidate.get("kind", ""))
        action_id = int(candidate.get("action_id", -1))

        if kind == "simple":
            return [
                (("kind", "simple"), 0.35),
                (("aid", action_id), 0.65),
            ]

        if kind == "click":
            source = str(feats.get("source", "click"))
            group = int(candidate.get("group", feats.get("group", 4)))
            color = int(feats.get("color", -1))
            area_bucket = int(feats.get("area_bucket", -1))
            rect = int(feats.get("rect", 0))
            edge_mask = int(feats.get("edge_mask", 0))
            rep_idx = int(feats.get("rep_idx", -1))
            bin_x = int(feats.get("bin_x", -1))
            bin_y = int(feats.get("bin_y", -1))
            return [
                (("kind", "click"), 0.10),
                (("aid", action_id), 0.16),
                (("group", group), 0.16),
                (("source", source), 0.10),
                (("color", color), 0.14),
                (("group_color", group, color), 0.10),
                (("bin", bin_x, bin_y), 0.10),
                (("rep", rep_idx), 0.04),
                (("area", area_bucket), 0.05),
                (("rect", rect), 0.03),
                (("edge", edge_mask), 0.02),
            ]

        return [(("kind", kind or "other"), 1.0)]

    def estimate(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        total_w = sum(float(w) for _, w in weighted)
        total_v = sum(float(w) * self.mean(key) for key, w in weighted)
        return float(total_v / total_w) if total_w > 0 else 0.5

    def exposure(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        total_w = sum(float(w) for _, w in weighted)
        total_v = sum(float(w) * self.count(key) for key, w in weighted)
        return float(total_v / total_w) if total_w > 0 else 0.0

    def update_candidate(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if level_up:
            success_inc = 1.30
            failure_inc = 0.00
        elif changed and not suspicious:
            success_inc = 0.70
            failure_inc = 0.30
        elif changed and suspicious:
            success_inc = 0.20
            failure_inc = 0.80
        elif game_over or errored:
            success_inc = 0.00
            failure_inc = 1.20
        else:
            success_inc = 0.00
            failure_inc = 1.00

        for key, _ in self._candidate_weighted_keys(candidate):
            self.update(key, success_inc, failure_inc)


@dataclass
class FrameView:
    node_id: str
    hash_id: str
    raw_frame: np.ndarray
    num_frames: int
    state: GameState
    levels_completed: int
    available_action_ids: set[int]
    segments: list[dict[str, Any]]
    candidates: list[dict[str, Any]]
    groups: list[set[int]]
    status_mask: np.ndarray
    diff_mask: np.ndarray


class FrameProcessor:
    OFFSETS4 = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self) -> None:
        self.frame_shape = (64, 64)
        self.status_bar_color = 16
        self.salient_colors = {6, 7, 8, 9, 10, 11, 12, 13, 14, 15}
        self.status_bar_ratio_threshold = 5.0
        self.status_bar_twins_threshold = 3

    def _set_shape(self, shape: tuple[int, int]) -> None:
        self.frame_shape = tuple(int(v) for v in shape)

    @property
    def status_bar_distance_threshold(self) -> int:
        return max(1, min(3, min(self.frame_shape) // 12 + 1))

    @property
    def minimal_width(self) -> int:
        return 2 if min(self.frame_shape) >= 8 else 1

    @property
    def maximal_width(self) -> int:
        return max(2, min(self.frame_shape) // 2)

    def segment_frame(self, frame: np.ndarray) -> tuple[np.ndarray, list[dict[str, Any]]]:
        frame = np.asarray(frame, dtype=np.uint8)
        self._set_shape(tuple(frame.shape))
        h, w = frame.shape
        label_map = np.full((h, w), -1, dtype=np.int32)
        components: list[dict[str, Any]] = []
        cid = -1

        for y in range(h):
            for x in range(w):
                if label_map[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                label_map[y, x] = cid

                min_x = max_x = x
                min_y = max_y = y
                area = 0
                points: list[tuple[int, int]] = []

                while q:
                    cy, cx = q.popleft()
                    points.append((cy, cx))
                    area += 1
                    min_x = min(min_x, cx)
                    max_x = max(max_x, cx)
                    min_y = min(min_y, cy)
                    max_y = max(max_y, cy)
                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and label_map[ny, nx] == -1 and int(frame[ny, nx]) == color:
                            label_map[ny, nx] = cid
                            q.append((ny, nx))

                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                components.append(
                    {
                        "bounding_box": (min_x, min_y, max_x, max_y),
                        "color": color,
                        "area": area,
                        "is_rectangle": area == rect_area,
                        "points": points,
                    }
                )

        buckets: dict[tuple[int, bool, int], list[int]] = defaultdict(list)
        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            buckets[key].append(i)

        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            twins = [j for j in buckets[key] if j != i]
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins

        return label_map, components

    def check_segment_fully_on_edge(self, segment: dict[str, Any], edges: Optional[list[str]] = None) -> list[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        thr = self.status_bar_distance_threshold
        if edges is None:
            edges = ["any"]
        result = []
        if "left" in edges or "any" in edges:
            if max(x1, x2) < thr:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) >= self.frame_shape[1] - thr:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < thr:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) >= self.frame_shape[0] - thr:
                result.append("bottom")
        return result

    def check_segment_ratio(self, segment: dict[str, Any], direction: str = "any") -> bool:
        x1, y1, x2, y2 = segment["bounding_box"]
        x_len = x2 - x1 + 1
        y_len = y2 - y1 + 1
        ratio = x_len / max(y_len, 1)
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1.0 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def segment_twins_on_edge(self, segment: dict[str, Any], frame_segments: list[dict[str, Any]], edges: Optional[list[str]] = None) -> list[int]:
        if edges is None:
            edges = self.check_segment_fully_on_edge(segment, ["any"])
            if not edges:
                return []
        twins = []
        for twin_id in segment.get("twin_ids", []):
            twin = frame_segments[int(twin_id)]
            twin_edges = self.check_segment_fully_on_edge(twin, edges)
            if twin_edges:
                twins.append(int(twin_id))
        return twins

    def identify_status_bars(self, segmented_frame: np.ndarray, frame_segments: list[dict[str, Any]]) -> np.ndarray:
        checked_segment_ids = set()
        status_bar_segment_ids_list: list[list[int]] = []

        for i, segment in enumerate(frame_segments):
            if i in checked_segment_ids:
                continue
            checked_segment_ids.add(i)
            on_edge_list = self.check_segment_fully_on_edge(segment, edges=["any"])
            if not on_edge_list:
                continue

            directions = []
            if "left" in on_edge_list or "right" in on_edge_list:
                directions.append("vertical")
            if "top" in on_edge_list or "bottom" in on_edge_list:
                directions.append("horizontal")
            direction = "any" if len(directions) == 2 else directions[0]

            status_bar_segment_ids = [i]
            is_long_ratio = self.check_segment_ratio(segment, direction=direction)

            if not is_long_ratio:
                twin_ids_on_edge = self.segment_twins_on_edge(segment, frame_segments)
                for twin_id in twin_ids_on_edge:
                    checked_segment_ids.add(int(twin_id))
                if len(twin_ids_on_edge) + 1 < self.status_bar_twins_threshold:
                    continue
                status_bar_segment_ids.extend(int(t) for t in twin_ids_on_edge)

            status_bar_segment_ids_list.append(status_bar_segment_ids)

        status_bar_mask = np.zeros(segmented_frame.shape, dtype=bool)
        for status_bar_segment_ids in status_bar_segment_ids_list:
            for status_bar_segment_id in status_bar_segment_ids:
                status_bar_mask[segmented_frame == int(status_bar_segment_id)] = True
        return status_bar_mask

    def hash_frame(self, frame: np.ndarray) -> str:
        frame = np.asarray(frame, dtype=np.uint8, order="C")
        flat = frame.ravel()
        if flat.size % 2 == 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        return hashlib.blake2b(
            packed.tobytes(),
            digest_size=16,
            person=repr(frame.shape).encode(),
        ).hexdigest()

    def segment_group(self, segment: dict[str, Any]) -> int:
        x1, y1, x2, y2 = segment["bounding_box"]
        xw = x2 - x1 + 1
        yw = y2 - y1 + 1
        color = int(segment["color"])
        is_salient = color in self.salient_colors
        is_medium = self.minimal_width <= xw <= self.maximal_width and self.minimal_width <= yw <= self.maximal_width
        is_status_bar = color == self.status_bar_color

        if is_salient and is_medium:
            return 0
        if is_medium:
            return 1
        if is_salient:
            return 2
        if not is_status_bar:
            return 3
        return 4

    def representative_points(self, segment: dict[str, Any]) -> list[tuple[int, int]]:
        pts = segment.get("points", [])
        if not pts:
            return []
        x1, y1, x2, y2 = segment["bounding_box"]
        cy = 0.5 * (y1 + y2)
        cx = 0.5 * (x1 + x2)

        def dist2(p: tuple[int, int]) -> float:
            py, px = p
            return (py - cy) ** 2 + (px - cx) ** 2

        center_point = min(pts, key=dist2)
        first_point = min(pts)
        last_point = max(pts)

        out = [center_point]
        if len(pts) >= 12:
            out.append(first_point)
        if len(pts) >= 32:
            out.append(last_point)

        dedup = []
        seen = set()
        for py, px in out:
            key = (int(px), int(py))
            if key not in seen:
                seen.add(key)
                dedup.append((int(px), int(py)))
        return dedup

    def dominant_color(self, frame: np.ndarray, invalid_mask: Optional[np.ndarray] = None) -> int:
        frame = np.asarray(frame, dtype=np.uint8)
        if invalid_mask is None:
            vals = frame.ravel()
        else:
            keep = ~np.asarray(invalid_mask, dtype=bool)
            vals = frame[keep]
            if vals.size == 0:
                vals = frame.ravel()
        counts = np.bincount(vals.astype(np.int64), minlength=17)
        return int(np.argmax(counts[:16]))

    def segment_priority(self, segment: dict[str, Any], background_color: int) -> float:
        x1, y1, x2, y2 = segment["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        area = int(segment["area"])
        color = int(segment["color"])

        score = 0.0
        if color != background_color:
            score += 0.80
        if color in self.salient_colors:
            score += 1.00
        if self.minimal_width <= width <= self.maximal_width and self.minimal_width <= height <= self.maximal_width:
            score += 0.65
        if bool(segment.get("is_rectangle")):
            score += 0.15
        if color == self.status_bar_color:
            score -= 2.0
        score -= 0.0125 * min(area, 100)
        return float(score)

    def diff_points(self, prev_frame: Optional[np.ndarray], curr_frame: np.ndarray, invalid_mask: np.ndarray, max_points: int = 6) -> list[tuple[int, int]]:
        if prev_frame is None:
            return []
        prev_frame = np.asarray(prev_frame, dtype=np.uint8)
        curr_frame = np.asarray(curr_frame, dtype=np.uint8)
        if prev_frame.shape != curr_frame.shape:
            return []

        diff = (prev_frame != curr_frame) & (~np.asarray(invalid_mask, dtype=bool))
        if not np.any(diff):
            return []

        h, w = diff.shape
        visited = np.zeros((h, w), dtype=bool)
        comps: list[tuple[int, tuple[int, int]]] = []

        for y in range(h):
            for x in range(w):
                if not diff[y, x] or visited[y, x]:
                    continue
                q = deque([(y, x)])
                visited[y, x] = True
                pts: list[tuple[int, int]] = []
                while q:
                    cy, cx = q.popleft()
                    pts.append((cy, cx))
                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and diff[ny, nx] and not visited[ny, nx]:
                            visited[ny, nx] = True
                            q.append((ny, nx))
                cy = sum(py for py, _ in pts) / len(pts)
                cx = sum(px for _, px in pts) / len(pts)
                py, px = min(pts, key=lambda p: (p[0] - cy) ** 2 + (p[1] - cx) ** 2)
                comps.append((len(pts), (int(px), int(py))))

        comps.sort(key=lambda item: (-item[0], item[1][1], item[1][0]))
        seen: set[tuple[int, int]] = set()
        out: list[tuple[int, int]] = []
        for _, pt in comps:
            if pt in seen:
                continue
            seen.add(pt)
            out.append(pt)
            if len(out) >= max_points:
                break
        return out

    def edge_mask(self, segment: dict[str, Any]) -> int:
        mask = 0
        for edge in self.check_segment_fully_on_edge(segment, ["any"]):
            if edge == "left":
                mask |= 1
            elif edge == "right":
                mask |= 2
            elif edge == "top":
                mask |= 4
            elif edge == "bottom":
                mask |= 8
        return int(mask)

    def area_bucket(self, area: int) -> int:
        return int(min(7, max(0, int(math.log2(max(1, int(area)))))))

    def candidate_features(
        self,
        *,
        kind: str,
        action_id: int,
        group: int,
        x: Optional[int] = None,
        y: Optional[int] = None,
        raw_frame: Optional[np.ndarray] = None,
        segment: Optional[dict[str, Any]] = None,
        source: str = "simple",
        rep_idx: int = 0,
    ) -> dict[str, Any]:
        out: dict[str, Any] = {
            "kind": str(kind),
            "action_id": int(action_id),
            "group": int(group),
            "source": str(source),
        }
        if kind == "simple":
            return out

        h, w = raw_frame.shape if raw_frame is not None else self.frame_shape
        xi = int(0 if x is None else x)
        yi = int(0 if y is None else y)
        out["x"] = xi
        out["y"] = yi
        out["bin_x"] = int(min(3, (4 * xi) // max(1, w - 1))) if w > 1 else 0
        out["bin_y"] = int(min(3, (4 * yi) // max(1, h - 1))) if h > 1 else 0
        out["rep_idx"] = int(rep_idx)

        if segment is not None:
            out["color"] = int(segment["color"])
            out["area_bucket"] = self.area_bucket(int(segment["area"]))
            out["rect"] = int(bool(segment["is_rectangle"]))
            out["edge_mask"] = self.edge_mask(segment)
        else:
            color = int(raw_frame[yi, xi]) if raw_frame is not None else -1
            out["color"] = color
            out["area_bucket"] = 0
            out["rect"] = 1
            edge_mask = 0
            if xi <= 1:
                edge_mask |= 1
            if xi >= w - 2:
                edge_mask |= 2
            if yi <= 1:
                edge_mask |= 4
            if yi >= h - 2:
                edge_mask |= 8
            out["edge_mask"] = int(edge_mask)
        return out

    def coarse_grid_points(self, frame: np.ndarray, invalid_mask: np.ndarray, grid_n: int = 4) -> list[tuple[int, int]]:
        h, w = frame.shape
        ys = np.linspace(0, h - 1, grid_n).round().astype(int).tolist()
        xs = np.linspace(0, w - 1, grid_n).round().astype(int).tolist()

        points: list[tuple[int, int]] = []
        seen = set()
        for y in ys:
            for x in xs:
                if not invalid_mask[y, x]:
                    key = (x, y)
                    if key not in seen:
                        seen.add(key)
                        points.append(key)
                    continue
                # small local search around blocked grid points
                found = False
                for radius in (1, 2, 3):
                    for dy in range(-radius, radius + 1):
                        for dx in range(-radius, radius + 1):
                            ny = int(max(0, min(h - 1, y + dy)))
                            nx = int(max(0, min(w - 1, x + dx)))
                            if not invalid_mask[ny, nx]:
                                key = (nx, ny)
                                if key not in seen:
                                    seen.add(key)
                                    points.append(key)
                                found = True
                                break
                        if found:
                            break
                    if found:
                        break
        return points


class MyAgent(Agent):
    MAX_ACTIONS = 720
    N_GROUPS = 5
    TOTAL_TIME_ALLOWED = 5.75 * 60 * 60

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        self.py_rng = random.Random(seed)
        self.np_rng = np.random.default_rng(seed % (2**32 - 1))

        self.fp = FrameProcessor()
        self.graph = GraphExplorer(n_groups=self.N_GROUPS)
        self.level_memory = FeatureMemory()
        self.game_memory = FeatureMemory()

        self.current_level: Optional[int] = None
        self.status_bar_mask: Optional[np.ndarray] = None
        self.start_node: Optional[str] = None
        self.last_view: Optional[FrameView] = None
        self.last_candidate_idx: Optional[int] = None
        self.last_success_signature: Optional[tuple[Any, ...]] = None
        self.last_unique_nodes = 0
        self.steps_in_level = 0
        self.resets_in_level = 0
        self.steps_since_new_node = 0
        self.steps_since_hash_change = 0
        self.start_time = time.time()

    def reset_level_state(self) -> None:
        self.level_memory = FeatureMemory()
        self.status_bar_mask = None
        self.start_node = None
        self.graph.reset()
        self.steps_in_level = 0
        self.resets_in_level = 0
        self.steps_since_new_node = 0
        self.steps_since_hash_change = 0
        self.last_unique_nodes = 0
        self.last_view = None
        self.last_candidate_idx = None

    def _candidate_signature(self, candidate: dict[str, Any]) -> Optional[tuple[Any, ...]]:
        feats = candidate.get("features") or {}
        if candidate.get("kind") == "simple":
            return ("simple", int(candidate.get("action_id", -1)))
        if candidate.get("kind") == "click":
            return (
                "click",
                int(feats.get("bin_x", -1)),
                int(feats.get("bin_y", -1)),
                int(feats.get("color", -1)),
                int(candidate.get("group", -1)),
                str(feats.get("source", "")),
            )
        return None

    def remember(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        self.level_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.game_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        if changed:
            sig = self._candidate_signature(candidate)
            if sig is not None:
                self.last_success_signature = sig

    def candidate_priority(self, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        local_est = self.level_memory.estimate(candidate)
        global_est = self.game_memory.estimate(candidate)
        prior = 0.62 * local_est + 0.38 * global_est

        exposure = 0.70 * self.level_memory.exposure(candidate) + 0.30 * self.game_memory.exposure(candidate)
        novelty = 1.0 / math.sqrt(1.0 + exposure)

        feats = candidate.get("features") or {}
        group = int(candidate.get("group", 0))
        source = str(feats.get("source", ""))

        score = 0.72 * prior + 0.22 * novelty
        score -= 0.05 * float(group)

        if source == "diff":
            score += 0.08
        elif source == "segment":
            score += 0.03
        elif source == "grid":
            score -= 0.02

        if candidate.get("kind") == "simple":
            if self.steps_in_level < 10:
                score += 0.10
            if self.current_level is not None and self.current_level >= 1:
                score += 0.03

        sig = self._candidate_signature(candidate)
        if sig is not None and sig == self.last_success_signature:
            score += 0.06

        if path_mode:
            score -= 0.14 * float(distance)

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def _pick_topk(self, candidate_ids: list[int], score_fn: Any, *, temperature: float, top_k: int) -> int:
        ranked = sorted(candidate_ids, key=score_fn, reverse=True)
        ranked = ranked[: max(1, min(top_k, len(ranked)))]
        if len(ranked) == 1:
            return int(ranked[0])

        vals = np.asarray([float(score_fn(idx)) for idx in ranked], dtype=np.float64)
        vals = vals - vals.max()
        vals = vals / max(temperature, 1e-3)
        probs = np.exp(vals)
        probs_sum = probs.sum()
        if not np.isfinite(probs_sum) or probs_sum <= 0:
            return int(ranked[0])
        probs = probs / probs_sum
        return int(self.np_rng.choice(ranked, p=probs))

    def _normalize_available_action_ids(self, latest_frame: FrameData) -> set[int]:
        raw_actions = getattr(latest_frame, "available_actions", []) or []
        out = set()
        for a in raw_actions:
            aid = normalize_action_id(a)
            if aid is not None:
                out.add(int(aid))
        return out

    def _build_view(self, latest_frame: FrameData, prev_raw_frame: Optional[np.ndarray] = None) -> FrameView:
        raw_frame, num_frames = extract_latest_grid(latest_frame)
        available_action_ids = self._normalize_available_action_ids(latest_frame)

        if self.status_bar_mask is None or tuple(np.asarray(self.status_bar_mask).shape) != tuple(raw_frame.shape):
            initial_labels, initial_segments = self.fp.segment_frame(raw_frame)
            self.status_bar_mask = self.fp.identify_status_bars(initial_labels, initial_segments)

        status_mask = np.asarray(self.status_bar_mask, dtype=bool).copy()
        diff_mask = np.zeros_like(raw_frame, dtype=bool)
        if prev_raw_frame is not None and np.asarray(prev_raw_frame).shape == raw_frame.shape:
            diff_mask = (np.asarray(prev_raw_frame, dtype=np.uint8) != raw_frame) & (~status_mask)

        masked_for_seg = raw_frame.copy()
        masked_for_seg[status_mask] = self.fp.status_bar_color
        hash_input = masked_for_seg.copy()
        hash_input[hash_input == self.fp.status_bar_color] = 0

        base_hash = self.fp.hash_frame(hash_input)
        action_sig = ",".join(str(a) for a in sorted(available_action_ids))
        node_id = f"{base_hash}|a:{action_sig}"

        labels, segments = self.fp.segment_frame(masked_for_seg)
        candidates: list[dict[str, Any]] = []
        candidate_groups: list[set[int]] = [set() for _ in range(self.N_GROUPS)]

        def add_simple(aid: int) -> None:
            cand_idx = len(candidates)
            candidates.append(
                {
                    "kind": "simple",
                    "action_id": int(aid),
                    "data": None,
                    "group": 0,
                    "tag": ("simple", int(aid)),
                    "features": self.fp.candidate_features(
                        kind="simple",
                        action_id=int(aid),
                        group=0,
                        source="simple",
                    ),
                }
            )
            candidate_groups[0].add(cand_idx)

        def add_click(
            x: int,
            y: int,
            group: int,
            tag: tuple[Any, ...],
            source: str,
            segment: Optional[dict[str, Any]] = None,
            rep_idx: int = 0,
        ) -> None:
            h, w = raw_frame.shape
            xi = int(max(0, min(w - 1, x)))
            yi = int(max(0, min(h - 1, y)))
            if status_mask[yi, xi]:
                return
            cand_idx = len(candidates)
            group_i = int(max(0, min(self.N_GROUPS - 1, group)))
            candidates.append(
                {
                    "kind": "click",
                    "action_id": 6,
                    "data": {"x": xi, "y": yi},
                    "group": group_i,
                    "tag": tag,
                    "features": self.fp.candidate_features(
                        kind="click",
                        action_id=6,
                        group=group_i,
                        x=xi,
                        y=yi,
                        raw_frame=raw_frame,
                        segment=segment,
                        source=source,
                        rep_idx=rep_idx,
                    ),
                }
            )
            candidate_groups[group_i].add(cand_idx)

        any_simple_actions = False
        for aid in sorted(available_action_ids):
            if aid in SIMPLE_ACTION_IDS:
                add_simple(int(aid))
                any_simple_actions = True

        if 6 in available_action_ids:
            background_color = self.fp.dominant_color(raw_frame, invalid_mask=status_mask)
            ordered_segments = sorted(
                enumerate(segments),
                key=lambda kv: self.fp.segment_priority(kv[1], background_color),
                reverse=True,
            )

            used_points: set[tuple[int, int]] = set()
            for seg_rank, (seg_id, seg) in enumerate(ordered_segments[:18]):
                color = int(seg["color"])
                if color == self.fp.status_bar_color:
                    continue
                reps = self.fp.representative_points(seg)
                if not reps:
                    continue
                max_reps = 2 if seg_rank < 8 else 1
                for rep_idx, (x, y) in enumerate(reps[:max_reps]):
                    if (x, y) in used_points:
                        continue
                    used_points.add((x, y))
                    group = self.fp.segment_group(seg)
                    if not any_simple_actions and seg_rank < 3 and rep_idx == 0:
                        group = 0
                    elif rep_idx > 0:
                        group = min(self.N_GROUPS - 1, group + 1)
                    add_click(x, y, group, ("seg", int(seg_id), int(rep_idx), int(x), int(y)), "segment", seg, rep_idx)

            for j, (x, y) in enumerate(self.fp.diff_points(prev_raw_frame, raw_frame, status_mask, max_points=6)):
                if (x, y) in used_points:
                    continue
                used_points.add((x, y))
                group = 0 if j < 3 else 1
                add_click(x, y, group, ("diff", j, int(x), int(y)), "diff", None, 0)

            invalid = status_mask.copy()
            for x, y in used_points:
                invalid[int(y), int(x)] = True
            for j, (x, y) in enumerate(self.fp.coarse_grid_points(raw_frame, invalid_mask=invalid, grid_n=4)[:8]):
                if (x, y) in used_points:
                    continue
                used_points.add((x, y))
                if not any_simple_actions and j < 2:
                    group = 0
                else:
                    group = self.N_GROUPS - 2 if j < 4 else self.N_GROUPS - 1
                add_click(x, y, group, ("grid", j, int(x), int(y)), "grid", None, 0)

        return FrameView(
            node_id=node_id,
            hash_id=base_hash,
            raw_frame=raw_frame,
            num_frames=num_frames,
            state=latest_frame.state,
            levels_completed=int(getattr(latest_frame, "levels_completed", 0) or 0),
            available_action_ids=set(available_action_ids),
            segments=segments,
            candidates=candidates,
            groups=candidate_groups,
            status_mask=status_mask,
            diff_mask=diff_mask,
        )

    def _level_budget(self, level_idx: int) -> int:
        table = {
            1: 96,
            2: 140,
            3: 210,
            4: 290,
            5: 380,
            6: 480,
            7: 600,
        }
        return table.get(int(level_idx), 680)

    def _max_resets(self, level_idx: int) -> int:
        if level_idx <= 2:
            return 6
        if level_idx <= 4:
            return 8
        return 10

    def _stagnation_limits(self, level_idx: int) -> tuple[int, int]:
        if level_idx <= 2:
            return 18, 12
        if level_idx <= 4:
            return 26, 18
        return 34, 24

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        level_idx = max(1, int(view.levels_completed) + 1)
        budget = self._level_budget(level_idx)
        max_resets = self._max_resets(level_idx)
        new_node_limit, hash_change_limit = self._stagnation_limits(level_idx)

        if self.steps_in_level >= budget:
            if self.resets_in_level < max_resets:
                return {"kind": "meta_reset", "reasoning": f"budget reset at level {level_idx}"}
            return None

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable or (stagnated and view.node_id != self.start_node and self.resets_in_level < max_resets):
            return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        def score_open(idx: int) -> float:
            return self.candidate_priority(view.candidates[idx], path_mode=False)

        def score_path(idx: int) -> float:
            return self.candidate_priority(
                view.candidates[idx],
                path_mode=True,
                distance=self.graph.edge_distance(view.node_id, idx),
            )

        open_ids = self.graph.open_candidate_ids(view.node_id)
        edge_idx: Optional[int] = None

        if open_ids:
            early_simple = [idx for idx in open_ids if view.candidates[idx].get("kind") == "simple"]
            if early_simple and self.steps_in_level < min(10, len(view.candidates) + 2):
                edge_idx = self._pick_topk(
                    early_simple,
                    score_open,
                    temperature=0.40,
                    top_k=min(4, len(early_simple)),
                )
            else:
                edge_idx = self._pick_topk(
                    open_ids,
                    score_open,
                    temperature=0.26 if self.steps_since_hash_change < 6 else 0.42,
                    top_k=min(8, len(open_ids)),
                )
        else:
            path_ids = self.graph.successful_candidate_ids(view.node_id)
            if not path_ids:
                if view.node_id != self.start_node and self.resets_in_level < max_resets:
                    return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
                return None
            edge_idx = self._pick_topk(
                path_ids,
                score_path,
                temperature=0.20,
                top_k=min(5, len(path_ids)),
            )

        cand = dict(view.candidates[int(edge_idx)])
        cand["candidate_idx"] = int(edge_idx)
        return cand

    def _action_from_candidate(self, choice: dict[str, Any]) -> GameAction:
        action = GameAction.from_id(int(choice["action_id"]))
        if choice["kind"] == "click":
            action.set_data(
                {
                    "x": int(choice["data"]["x"]),
                    "y": int(choice["data"]["y"]),
                }
            )
        tag = choice.get("tag")
        if isinstance(tag, tuple):
            tag = list(tag)
        action.reasoning = {
            "kind": str(choice["kind"]),
            "tag": tag,
            "group": int(choice.get("group", 0)),
        }
        return action

    def _apply_previous_transition(self, current_view: FrameView) -> None:
        if self.last_view is None or self.last_candidate_idx is None:
            return

        prev_candidate = self.last_view.candidates[self.last_candidate_idx]
        changed = current_view.node_id != self.last_view.node_id
        pixel_change = float(np.asarray(current_view.diff_mask, dtype=bool).mean()) if current_view.diff_mask.size else 0.0
        suspicious = bool(
            changed
            and self.start_node is not None
            and current_view.node_id == self.start_node
            and current_view.num_frames > 1
            and pixel_change > 0.0
        )

        self.graph.record_test(
            self.last_view.node_id,
            self.last_candidate_idx,
            1 if changed else 0,
            target_node=current_view.node_id if changed else None,
            target_num_candidates=len(current_view.candidates),
            group2remaining_candidate_ids=current_view.groups,
            suspicious_transition=suspicious,
        )
        self.remember(prev_candidate, changed=changed, suspicious=suspicious)

        node_count = len(self.graph._nodes)
        if node_count > self.last_unique_nodes:
            self.steps_since_new_node = 0
            self.last_unique_nodes = node_count
        else:
            self.steps_since_new_node += 1

        if changed:
            self.steps_since_hash_change = 0
        else:
            self.steps_since_hash_change += 1

        self.last_view = None
        self.last_candidate_idx = None

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if latest_frame.state == GameState.WIN:
            return True
        if self.action_counter >= self.MAX_ACTIONS:
            return True
        if (time.time() - self.start_time) >= self.TOTAL_TIME_ALLOWED:
            return True
        return False

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        # Start or restart a level if needed.
        if latest_frame.state == GameState.NOT_PLAYED:
            self.last_view = None
            self.last_candidate_idx = None
            action = GameAction.RESET
            action.reasoning = "RESET to start or restart the current level"
            return action

        curr_level = int(getattr(latest_frame, "levels_completed", 0) or 0)
        level_changed = self.current_level is None or curr_level != self.current_level

        # If the previous action produced GAME_OVER, mark it and reset.
        if latest_frame.state == GameState.GAME_OVER:
            if self.last_view is not None and self.last_candidate_idx is not None:
                prev_candidate = self.last_view.candidates[self.last_candidate_idx]
                self.graph.record_test(self.last_view.node_id, self.last_candidate_idx, 0, None)
                self.remember(prev_candidate, changed=False, game_over=True)
                self.last_view = None
                self.last_candidate_idx = None
            self.resets_in_level += 1
            self.steps_in_level += 1
            action = GameAction.RESET
            action.reasoning = "RESET after GAME_OVER"
            return action

        # If we just completed a level, reward the action that led here and reset per-level state.
        if level_changed and self.current_level is not None and curr_level > self.current_level:
            if self.last_view is not None and self.last_candidate_idx is not None:
                prev_candidate = self.last_view.candidates[self.last_candidate_idx]
                self.remember(prev_candidate, changed=True, level_up=True)
            self.reset_level_state()
            self.current_level = curr_level

        # First active frame of a level or post-reset recovery.
        if self.current_level is None:
            self.current_level = curr_level
            self.reset_level_state()

        prev_raw = self.last_view.raw_frame if self.last_view is not None else None
        current_view = self._build_view(latest_frame, prev_raw_frame=prev_raw)

        if self.start_node is None or level_changed:
            self.start_node = current_view.node_id
            self.graph.initialize(current_view.node_id, len(current_view.candidates), current_view.groups)
            self.last_unique_nodes = len(self.graph._nodes)
        else:
            if not self.graph.has_node(current_view.node_id):
                self.graph.ensure_node(current_view.node_id, len(current_view.candidates), current_view.groups)
                self.last_unique_nodes = len(self.graph._nodes)

        # Record the outcome of the prior candidate action now that we can see the new state.
        self._apply_previous_transition(current_view)

        choice = self.choose_candidate(current_view)

        if choice is None:
            if self.resets_in_level < self._max_resets(max(1, curr_level + 1)):
                self.resets_in_level += 1
                self.steps_in_level += 1
                action = GameAction.RESET
                action.reasoning = "RESET because no useful candidate is reachable"
                return action
            fallback_ids = sorted([aid for aid in current_view.available_action_ids if aid in SIMPLE_ACTION_IDS])
            if 6 in current_view.available_action_ids and current_view.candidates:
                click_candidates = [c for c in current_view.candidates if c.get("kind") == "click"]
                if click_candidates:
                    choice = dict(max(click_candidates, key=lambda c: self.candidate_priority(c)))
                    action = self._action_from_candidate(choice)
                    self.steps_in_level += 1
                    return action
            if fallback_ids:
                action = GameAction.from_id(int(fallback_ids[0]))
            else:
                action = GameAction.RESET
            action.reasoning = "Fallback action"
            self.steps_in_level += 1
            return action

        if choice["kind"] == "meta_reset":
            self.last_view = None
            self.last_candidate_idx = None
            self.resets_in_level += 1
            self.steps_in_level += 1
            action = GameAction.RESET
            action.reasoning = str(choice.get("reasoning", "strategic reset"))
            return action

        action = self._action_from_candidate(choice)
        self.last_view = current_view
        self.last_candidate_idx = int(choice["candidate_idx"])
        self.steps_in_level += 1
        return action


Writing /kaggle/working/my_agent.py


In [3]:
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import requests

SUBMISSION_PATH = Path("/kaggle/working/submission.parquet")


def make_dummy_submission() -> pd.DataFrame:
    df = pd.DataFrame(
        data=[["0_0", "0", False, 0.0]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    df.to_parquet(SUBMISSION_PATH, index=False)
    return df


def load_submission() -> pd.DataFrame:
    candidate_paths = [
        SUBMISSION_PATH,
        Path("/kaggle/working/ARC-AGI-3-Agents/submission.parquet"),
    ]
    for path in candidate_paths:
        if path.exists():
            return pd.read_parquet(path)

    hits = sorted(Path("/kaggle/working").rglob("submission.parquet"))
    if hits:
        return pd.read_parquet(hits[0])

    return make_dummy_submission()


def wait_for_gateway(url: str = "http://gateway:8001/api/games", timeout_s: int = 600) -> None:
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=10)
            if response.ok:
                return
            last_error = f"{response.status_code}: {response.text[:200]}"
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Gateway was not ready in time: {last_error}")


def find_competition_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
    ]
    for path in candidates:
        if path.exists():
            return path

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for repo_path in input_root.rglob("ARC-AGI-3-Agents"):
            return repo_path.parent

    raise FileNotFoundError("Could not locate the ARC-AGI-3 competition input directory.")


if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    wait_for_gateway()

    comp_root = find_competition_root()
    src_repo = comp_root / "ARC-AGI-3-Agents"
    dst_repo = Path("/kaggle/working/ARC-AGI-3-Agents")

    if dst_repo.exists():
        shutil.rmtree(dst_repo)
    shutil.copytree(src_repo, dst_repo)

    shutil.copy2("/kaggle/working/my_agent.py", dst_repo / "agents" / "templates" / "my_agent.py")

    (dst_repo / "agents" / "__init__.py").write_text(
        """from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""",
        encoding="utf-8",
    )

    (dst_repo / ".env").write_text(
        """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""",
        encoding="utf-8",
    )

    env = dict(os.environ)
    env["MPLBACKEND"] = "agg"

    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "myagent"],
        cwd=str(dst_repo),
        env=env,
        check=False,
    )
    print("main.py return code:", result.returncode)

    submission = load_submission()
    submission.to_parquet(SUBMISSION_PATH, index=False)
else:
    submission = make_dummy_submission()

In [4]:
print(submission.head(25))

  row_id game_id  end_of_game  score
0    0_0       0        False    0.0
